# DevOps on Databricks — CLI, Bundles & CI/CD with GitHub Actions

This notebook explains how the **dox_training** repo is built, deployed and promoted with the **Databricks CLI**, **Declarative Automation Bundles (DABs)** and **GitHub Actions**.

| # | Topic | What you will learn |
|---|---|---|
| 1 | Repo layout | Where code, resources and CI/CD files live |
| 2 | Databricks CLI — install & configure | Install, authentication types, profiles, env variables |
| 3 | Basic CLI commands | Workspace, compute, jobs, pipelines, UC, files, secrets |
| 4 | Bundle commands | `init`, `validate`, `deploy`, `run`, `summary`, `destroy` |
| 5 | `databricks.yml` explained | Bundle name, include, variables, targets, modes |
| 6 | Resource files explained | Pipelines and job YAML, `${...}` references |
| 7 | GitHub Actions workflow explained | Triggers, secrets, jobs, steps |
| 8 | Step-by-step CI/CD setup | Service principal → grants → GitHub secrets → first release |
| 9 | Daily developer flow | Feature branch → dev deploy → PR → prod |
| 10 | Troubleshooting & best practices | Common errors and fixes |

> The commands are shown as **text** — run them in a local terminal, the web terminal of a cluster, or a GitHub Actions runner. They are not meant to be executed as notebook cells.

**Key idea:** everything that is deployed (pipelines, jobs) is described **as code** in Git. A deploy = *"make the workspace look like the YAML in this commit"*.

## 1. Repo layout

```
dox_training/                              ← Git repo = bundle root
├── databricks.yml                         ← bundle definition: name, variables, targets (dev / prod)
├── resources/                             ← WHAT gets deployed (one YAML per resource)
│   ├── sdp_customer_pipeline_dlt.pipeline.yml
│   ├── sdp_customer_pipeline_dp.pipeline.yml
│   └── sdp_customer_pipelines.job.yml     ← runs dlt pipeline → dp pipeline
├── src/                                   ← the CODE the resources point to
│   ├── pipelines/
│   │   ├── sdp_customer_pipeline.py       (@dlt API)
│   │   ├── sdp_customer_pipeline_dp.py    (@dp API)
│   │   └── config/*.json                  (config-driven table names, sources, expectations)
│   ├── notebooks/                         ← training notebooks
│   └── devops                             ← this notebook
├── .github/workflows/databricks-bundle.yml ← CI/CD pipeline (GitHub Actions)
├── .gitignore                             ← ignores .databricks/ (local bundle state)
└── README.md
```

| Folder / file | Owner question it answers |
|---|---|
| `src/` | *What does the code do?* |
| `resources/` | *How is the code run?* (pipeline settings, job tasks, schedules, permissions) |
| `databricks.yml` | *Where is it deployed and with which values?* (targets, variables) |
| `.github/workflows/` | *When and by whom is it deployed?* (on PR, on merge to main, as a service principal) |

## 2. Databricks CLI — install & configure

The **Databricks CLI** (`databricks`) is a single binary that calls the Databricks REST APIs. Bundles are a sub-command of it (`databricks bundle ...`).

### 2.1 Install

| OS | Command |
|---|---|
| macOS (Homebrew) | `brew tap databricks/tap && brew install databricks` |
| Linux / macOS (script) | `curl -fsSL https://raw.githubusercontent.com/databricks/setup-cli/main/install.sh \| sh` |
| Windows | `winget install Databricks.DatabricksCLI` |
| GitHub Actions | `uses: databricks/setup-cli@main` |

```bash
databricks -v                 # check version (bundles need a recent version, e.g. 0.2xx+)
databricks --help             # list command groups
databricks jobs --help        # help for one group
```

### 2.2 Authentication types

| Type | Who uses it | How |
|---|---|---|
| **OAuth U2M** (user-to-machine) | Developers on a laptop | `databricks auth login` → browser login, token refreshed automatically |
| **OAuth M2M** (machine-to-machine) | CI/CD with a **service principal** | `DATABRICKS_CLIENT_ID` + `DATABRICKS_CLIENT_SECRET` |
| **Personal Access Token (PAT)** | Legacy / quick tests | `databricks configure --token` |
| **GitHub OIDC (workload identity federation)** | CI/CD without any stored secret | Federation policy on the service principal + `DATABRICKS_AUTH_TYPE=github-oidc` |

> Prefer **OAuth**. Avoid PATs for CI/CD — they belong to a person and expire / leak.

### 2.3 Configure a profile (developer laptop)

```bash
# OAuth login, saves a profile called "dox" in ~/.databrickscfg
databricks auth login --host https://adb-7405610410794404.4.azuredatabricks.net --profile dox

databricks auth profiles                  # list profiles and whether they are valid
databricks current-user me --profile dox  # who am I?
databricks auth env --profile dox         # show the resolved auth settings
```

`~/.databrickscfg` after login:

```ini
[dox]
host      = https://adb-7405610410794404.4.azuredatabricks.net
auth_type = databricks-cli

[dox-sp]                                  ; optional: service principal profile (OAuth M2M)
host          = https://adb-7405610410794404.4.azuredatabricks.net
client_id     = <application-id>
client_secret = <oauth-secret>
```

### 2.4 Environment variables (used by CI/CD)

| Variable | Meaning |
|---|---|
| `DATABRICKS_HOST` | Workspace URL |
| `DATABRICKS_CLIENT_ID` / `DATABRICKS_CLIENT_SECRET` | Service principal OAuth M2M credentials |
| `DATABRICKS_TOKEN` | PAT (if you use PAT auth) |
| `DATABRICKS_CONFIG_PROFILE` | Which profile in `~/.databrickscfg` to use by default |

The CLI resolves auth in this order: **command flags** (`--profile`) → **environment variables** → **`DEFAULT` profile** in `~/.databrickscfg`.

## 3. Basic CLI commands

Global flags that work everywhere: `--profile <name>`, `-o json` (JSON output), `--debug` (show HTTP calls), `--help`.

### Workspace & Git folders
```bash
databricks workspace list /Repos/training/dox_training
databricks workspace export "/Repos/training/dox_training/src/notebooks/Spark" --format SOURCE --file Spark.py
databricks workspace import "/Users/me@company.com/Spark" --file Spark.py --format SOURCE --language PYTHON
databricks repos list                                   # Git folders
databricks repos update <repo-id> --branch main         # pull latest main into a Git folder
```

### Compute
```bash
databricks clusters list
databricks clusters get 1102-204728-27elib3b
databricks clusters start 1102-204728-27elib3b
databricks warehouses list
```

### Jobs
```bash
databricks jobs list
databricks jobs get <job-id>
databricks jobs run-now <job-id>                        # waits until the run finishes
databricks jobs list-runs --job-id <job-id>
databricks jobs get-run <run-id>
```

### Pipelines (Lakeflow Spark Declarative Pipelines)
```bash
databricks pipelines list-pipelines
databricks pipelines get <pipeline-id>
databricks pipelines start-update <pipeline-id>                 # incremental refresh
databricks pipelines start-update <pipeline-id> --full-refresh  # rebuild all tables
databricks pipelines list-updates <pipeline-id>
```

### Unity Catalog
```bash
databricks catalogs list
databricks schemas list training
databricks tables list training default
databricks grants get table training.default.dlt_customer_silver
```

### Files in volumes
```bash
databricks fs ls dbfs:/Volumes/training/bronze/checkpoints
databricks fs cp ./orders.csv dbfs:/Volumes/training/bronze/landing/orders.csv
```

### Secrets
```bash
databricks secrets create-scope dox
databricks secrets put-secret dox api_key            # prompts for the value
databricks secrets list-secrets dox
# in code:  dbutils.secrets.get("dox", "api_key")
```

## 4. Bundle commands (Declarative Automation Bundles)

Run them from the **bundle root** (the folder with `databricks.yml`). `-t` selects the target.

| Command | What it does |
|---|---|
| `databricks bundle init` | Create a new bundle from a template (e.g. `default-python`) |
| `databricks bundle validate -t dev` | Check YAML, resolve variables and paths — **nothing is deployed** |
| `databricks bundle deploy -t dev` | Upload files and create / update / delete resources to match the YAML |
| `databricks bundle run -t dev sdp_customer_pipelines_job` | Run a deployed job or pipeline by its **resource key** |
| `databricks bundle run -t dev sdp_customer_pipeline_dp --full-refresh-all` | Full refresh of a pipeline |
| `databricks bundle summary -t dev` | Show deployed resources with IDs and URLs |
| `databricks bundle destroy -t dev` | Delete everything this bundle deployed in that target |
| `databricks bundle generate job --existing-job-id <id>` | Turn an existing UI-created job into YAML |
| `databricks bundle deployment bind <key> <id> -t prod` | Link a YAML resource to an existing workspace object |

**Override a variable at deploy time** (this is how we pass a different config file to a pipeline):

```bash
databricks bundle deploy -t dev --var="dp_config_file=sdp_pipeline_config_dp_test.json"
# or:  export BUNDLE_VAR_dp_config_file=sdp_pipeline_config_dp_test.json
```

**What a deploy does, step by step**
1. Reads `databricks.yml` + all files in `include:` and merges the selected target.
2. Resolves `${var.*}`, `${workspace.*}`, `${resources.*}` references.
3. Syncs files to `workspace.file_path` (in a Git folder in dev: *source-linked*, files are used in place).
4. Compares with the **state** in `workspace.state_path` and creates / updates / deletes resources.
5. Writes the new state — the next deploy knows what it owns.

## 5. `databricks.yml` explained

This is the actual file of this repo, with comments on each block.

```yaml
bundle:
  name: dox_training            # unique bundle name → part of deploy folder & state

include:
  - resources/*.yml             # merge every resource file into this bundle

variables:                      # values that differ per environment or per run
  catalog:
    default: training
  schema:
    default: default
  dlt_config_file:              # config file the @dlt pipeline reads
    default: sdp_pipeline_config_dlt.json
  dp_config_file:               # config file the @dp pipeline reads
    default: sdp_pipeline_config_dp.json

targets:                        # one entry per environment
  dev:
    mode: development           # see table below
    default: true               # used when -t is not given

  prod:
    mode: production
    workspace:
      root_path: /Workspace/Users/${workspace.current_user.userName}/.bundle/${bundle.name}/${bundle.target}
    permissions:                # applied to every resource in this target
      - level: CAN_MANAGE
        user_name: balaji.raghu@one-consultants.com
      - level: CAN_VIEW
        group_name: users
```

### Development vs production mode

| | `mode: development` | `mode: production` |
|---|---|---|
| Resource names | Prefixed `[dev <user>]` → every developer gets their own copy | Exact names |
| Pipelines | `development: true` (faster restarts, no retries) | `development: false` |
| Job schedules / triggers | **Paused** | Active |
| Deploy location | `~/.bundle/<bundle>/dev` of the deploying user | `root_path` you set |
| Deployment lock | Off | On (no two deploys at once) |
| Checks | Relaxed | Warns if e.g. `root_path` is writable by everyone, if Git branch differs |

### Useful built-in substitutions

| Expression | Example value |
|---|---|
| `${bundle.name}` / `${bundle.target}` | `dox_training` / `dev` |
| `${workspace.current_user.userName}` | `balaji.raghu@one-consultants.com` |
| `${workspace.file_path}` | Folder where the bundle files live after deploy |
| `${var.<name>}` | Value of a variable |
| `${resources.pipelines.<key>.id}` | ID of a pipeline deployed by this bundle |

### Typical additions for prod
```yaml
  prod:
    mode: production
    run_as:
      service_principal_name: <application-id>   # jobs/pipelines run as the SP, not as a person
    variables:
      schema: prod_customer                      # different schema per environment
```

## 6. Resource files explained (`resources/*.yml`)

Naming convention: `<name>.<resource_type>.yml`. Paths inside are **relative to the YAML file**, so `../src/...` points to the `src/` folder.

### Pipeline — `sdp_customer_pipeline_dp.pipeline.yml`
```yaml
resources:
  pipelines:
    sdp_customer_pipeline_dp:              # resource KEY (used by bundle run / references)
      name: sdp_customer_pipeline_dp       # display name (dev adds "[dev user] ")
      catalog: ${var.catalog}              # tables published to training.default
      schema: ${var.schema}
      serverless: true                     # no cluster config needed
      continuous: false                    # triggered (run by the job)
      channel: CURRENT
      root_path: ../src/pipelines
      libraries:
        - file:
            path: ../src/pipelines/sdp_customer_pipeline_dp.py
      configuration:                       # key/values readable with spark.conf.get(...)
        pipeline.config_path: ${workspace.file_path}/src/pipelines/config/${var.dp_config_file}
```

**How the config file reaches the code:** variable `dp_config_file` → pipeline `configuration` → `spark.conf.get("pipeline.config_path")` in `_load_config()`.

### Job — `sdp_customer_pipelines.job.yml`
```yaml
resources:
  jobs:
    sdp_customer_pipelines_job:
      name: sdp_customer_pipelines_job
      max_concurrent_runs: 1
      tasks:
        - task_key: run_sdp_customer_pipeline_dlt
          pipeline_task:
            pipeline_id: ${resources.pipelines.sdp_customer_pipeline_dlt.id}   # ID resolved at deploy
        - task_key: run_sdp_customer_pipeline_dp
          depends_on:
            - task_key: run_sdp_customer_pipeline_dlt                         # runs AFTER dlt
          pipeline_task:
            pipeline_id: ${resources.pipelines.sdp_customer_pipeline_dp.id}
```

> A `pipeline_task` cannot receive job parameters, which is why the config file name is a **bundle variable** instead of a job parameter.

Other things you can add to a job: `schedule` (cron), `trigger` (file arrival / table update), `email_notifications`, `parameters`, `notebook_task`, `python_wheel_task`, `sql_task`.

## 7. GitHub Actions workflow explained (`.github/workflows/databricks-bundle.yml`)

GitHub runs every YAML file in `.github/workflows/` on the events it lists.

```yaml
name: databricks-bundle                  # name shown in the GitHub "Actions" tab

on:                                      # WHEN it runs
  pull_request:
    branches: [main]                     # every PR into main → validate only
  push:
    branches: [main]                     # every merge/push to main → validate + deploy prod
  workflow_dispatch:                     # "Run workflow" button for manual runs

concurrency:                             # never run two deploys of the same branch at once
  group: databricks-bundle-${{ github.ref }}
  cancel-in-progress: false

env:                                     # credentials for the CLI, taken from repo secrets
  DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
  DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
  DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}

jobs:
  validate:                              # job 1 — always
    runs-on: ubuntu-latest               # fresh Linux VM from GitHub
    steps:
      - uses: actions/checkout@v4        # get the repo code
      - uses: databricks/setup-cli@main  # install the Databricks CLI
      - run: databricks bundle validate -t dev

  deploy-prod:                           # job 2 — only on main, after validate passed
    if: github.event_name != 'pull_request' && github.ref == 'refs/heads/main'
    needs: validate
    runs-on: ubuntu-latest
    environment: prod                    # GitHub environment → can require manual approval
    steps:
      - uses: actions/checkout@v4
      - uses: databricks/setup-cli@main
      - run: databricks bundle validate -t prod
      - run: databricks bundle deploy -t prod
      - run: databricks bundle run -t prod sdp_customer_pipelines_job   # smoke run
```

| Concept | Meaning |
|---|---|
| **Workflow** | The whole YAML file |
| **Event / trigger** (`on:`) | What starts it: PR, push, manual, schedule |
| **Job** | A group of steps on one runner VM; jobs run in parallel unless `needs:` |
| **Step** | One action (`uses:`) or shell command (`run:`) |
| **Secret** | Encrypted value, referenced as `${{ secrets.NAME }}`, masked in logs |
| **Environment** | Named stage (`prod`) with its own secrets, required reviewers, branch rules |

```
 feature branch ──PR──► [validate] ──✓──► review & merge ──push main──► [validate] ──► [approve] ──► [deploy prod] ──► [run job]
```

## 8. Step-by-step: set up CI/CD for this repo

### Step 1 — Create a service principal (SP)
1. Workspace **Settings → Identity and access → Service principals → Add service principal → Add new**.
2. Name it e.g. `sp-dox-training-cicd`. Copy its **Application ID**.
3. Entitlements: **Workspace access** (serverless jobs/pipelines need no cluster-create right).

### Step 2 — Create an OAuth secret for the SP
1. Open the SP → **Secrets** tab → **Generate secret**.
2. Copy the **secret** (shown once) and keep the **client ID** (= Application ID).

### Step 3 — Grant the SP access to data
Run as an admin (SQL):
```sql
GRANT USE CATALOG ON CATALOG training TO `<application-id>`;
GRANT USE SCHEMA, CREATE TABLE, CREATE MATERIALIZED VIEW ON SCHEMA training.default TO `<application-id>`;
GRANT READ FILES ON EXTERNAL LOCATION inboundfiles_location TO `<application-id>`;   -- Auto Loader source
```

### Step 4 — Make prod run as the SP (recommended)
In `databricks.yml` under `prod`:
```yaml
    run_as:
      service_principal_name: <application-id>
```
Now prod jobs and pipelines do not depend on a person's account.

### Step 5 — Add secrets in GitHub
Repo **Settings → Secrets and variables → Actions → New repository secret**:

| Secret | Value |
|---|---|
| `DATABRICKS_HOST` | `https://adb-7405610410794404.4.azuredatabricks.net` |
| `DATABRICKS_CLIENT_ID` | SP Application ID |
| `DATABRICKS_CLIENT_SECRET` | SP OAuth secret |

### Step 6 — Create the `prod` environment with approval
Repo **Settings → Environments → New environment → `prod`** → add **Required reviewers** and limit **Deployment branches** to `main`.

### Step 7 — Protect `main`
**Settings → Branches → Add rule** for `main`: require a pull request, require status check **`validate`** to pass.

### Step 8 — Test locally once
```bash
export DATABRICKS_HOST=https://adb-7405610410794404.4.azuredatabricks.net
export DATABRICKS_CLIENT_ID=<application-id>
export DATABRICKS_CLIENT_SECRET=<oauth-secret>
databricks current-user me          # should show the SP
databricks bundle validate -t prod
```

### Step 9 — First release
1. In the Databricks Git folder: **commit & push** the bundle files to a feature branch.
2. Open a **PR** → `validate` runs → review → **merge**.
3. The push to `main` starts `deploy-prod` → approve in GitHub → deploy → job run.
4. Check **Jobs & Pipelines** in Databricks: `sdp_customer_pipelines_job` (no `[dev]` prefix) owned/run by the SP.

### Step 10 — Rollback
Revert the bad commit in GitHub (`git revert`) and merge → the workflow redeploys the previous definition. Tables are not rolled back — use Delta `RESTORE` / time travel if data must be fixed.

> **No-secret option:** instead of Step 2/5 secrets, create a **GitHub OIDC federation policy** on the SP and set `DATABRICKS_AUTH_TYPE: github-oidc` + `permissions: id-token: write` in the workflow.

## 9. Daily developer flow

| Step | Where | Command / action |
|---|---|---|
| 1. Create a branch | Git folder → branch menu | `feature/add-orders-gold` |
| 2. Change code / YAML | `src/`, `resources/` | edit pipeline, config JSON, job |
| 3. Validate | Terminal / bundle panel | `databricks bundle validate -t dev` |
| 4. Deploy own copy | Terminal / bundle panel | `databricks bundle deploy -t dev` → `[dev you] ...` resources |
| 5. Test | Terminal / Jobs UI | `databricks bundle run -t dev sdp_customer_pipelines_job` |
| 6. Commit & push | Git folder | commit message, push branch |
| 7. Pull request | GitHub | CI runs `validate` → reviewer approves → merge |
| 8. Release | GitHub Actions | `deploy-prod` (after approval) |
| 9. Clean up | Terminal | `databricks bundle destroy -t dev` when the feature is done |

**Rules of thumb**
* Never edit prod jobs/pipelines in the UI — the next deploy overwrites them. Change the YAML instead.
* One bundle = one deployable unit. Keep secrets out of YAML and Git (use secret scopes / GitHub secrets).
* Parameterize anything environment-specific (`catalog`, `schema`, config file) with `variables`.

## 10. Troubleshooting & best practices

| Symptom | Likely cause | Fix |
|---|---|---|
| `cannot resolve bundle auth configuration` | No host / credentials for the CLI | Set `DATABRICKS_HOST` + client ID/secret, or `--profile` |
| `403 PERMISSION_DENIED` on deploy | SP lacks workspace or folder rights | Give the SP workspace access; check `root_path` permissions |
| Pipeline fails `PERMISSION_DENIED ... READ FILES` | SP has no access to the source | `GRANT READ FILES ON EXTERNAL LOCATION ...` |
| `notebook "..." not found` in validate | Wrong relative path / extension | Paths in `resources/` start with `../src/...`; notebooks need their extension |
| `Table ... is already managed by pipeline ...` | Two pipelines write the same table name | Different schema or table prefix (we use `dlt_` / `dp_`) |
| Deploy warns root path writable by all users | `root_path` under `/Workspace/Shared` | Use the deployer's home folder or restrict permissions |
| `deployment lock` error in prod | Another deploy is running or crashed | Wait, or `databricks bundle deploy --force-lock` |
| Prod resource changed in UI got reverted | Deploy overwrites UI edits | Edit the YAML in Git |

**Best practices**
1. `validate` on every PR, `deploy` only from `main`.
2. Deploy prod with a **service principal** (`run_as`) — never with a personal account.
3. Use **GitHub environments** with required reviewers for prod.
4. Keep environment differences in **targets + variables**, not in copied YAML files.
5. Use `mode: development` for personal testing — isolated names, paused schedules.
6. Store credentials only in **GitHub secrets** or **Databricks secret scopes**; prefer OIDC federation.
7. Pin the CLI version in CI when stability matters (`databricks/setup-cli@<version>`).